# RFM Customer Segmentation Analysis

In [ ]:
from datetime import datetime

# Analysis reference date（Calculate how many days ago this day was, considering it as the present day）
BASE_DATE = datetime(2026, 6, 30)

# Customer purchase log（dummy data）
# Include invalid data such as missing values (None) and negative amounts.
rfm_logs = [
    {"customer_id": "C001", "date": "2026-06-25", "amount": 5000},
    {"customer_id": "C002", "date": "2026-05-10", "amount": 12000},
    {"customer_id": "C001", "date": "2026-06-28", "amount": 4000},
    {"customer_id": "C003", "date": "2025-12-01", "amount": 30000},
    {"customer_id": "C004", "date": "2026-06-15", "amount": 15000},
    {"customer_id": "C002", "date": "2026-06-20", "amount": -2000}, # invalid data
    {"customer_id": "C005", "date": "2026-04-01", "amount": 8000},
    {"customer_id": "C001", "date": None,         "amount": 5000},  # invalid data
    {"customer_id": "C005", "date": "2026-05-20", "amount": 12000},
    {"customer_id": "C006", "date": "2026-06-29", "amount": 60000},
]

def filter_valid_logs(logs):
    """
    Filters out records with a missing date (None) or a non-positive amount (<= 0),
    and converts the date string to a datetime object.

    Args:
        logs (list): Customer purchase logs. Each log is a dictionary containing customer_id,
        date, and amount.

    Returns:
        list: A list of dictionaries containing customer_id, date(datetime object) and amount.
    """
    valid_logs= []
    for log in logs:
        if log["amount"] > 0 and log["date"] is not None:
            valid_logs.append({
                "customer_id": log["customer_id"],
                "date": datetime.strptime(log["date"], "%Y-%m-%d"),
                "amount": log["amount"]
            })

    return valid_logs

def aggregate_rfm(logs, base_date):
    """
    Aggregate RFM (R: Recency, F: Frequency, M: Monetary) data.
    
    Args:
        logs (list): Customer purchase logs. Each log is a dictionary containing customer_id,
        date, and amount.
        base_date (datetime): Reference date for calculating recency.
        
    Returns:
        list: A list of dictionaries containing customer_id, recency, frequency, and monetary.
    """
    valid_logs = filter_valid_logs(logs)
    customers = {}

    for log in valid_logs:
        cid = log["customer_id"]

        if cid not in customers:
            customers[cid] = {
                "latest_date": log["date"],
                "frequency": 0,
                "monetary": 0
            }

        customers[cid]["frequency"] += 1
        customers[cid]["monetary"] += log["amount"]

        if customers[cid]["latest_date"] < log["date"]:
            customers[cid]["latest_date"] = log["date"]
    
    return [
        {
            "customer_id": cid,
            "recency": (base_date - data["latest_date"]).days,
            "frequency": data["frequency"],
            "monetary": data["monetary"]
        }
        for cid, data in customers.items()
    ]

def score_rfm(logs, base_date):
    """
    Assign scores from 1 to 3 to the aggregated raw RFM data.
    
    Args:
        logs (list): Customer purchase logs. Each log is a dictionary containing customer_id,
        date, and amount.
        base_date (datetime): Reference date for calculating recency.
        
    Returns:
        list : A list of dictionaries containing customer_id, r_score, f_score, and m_score.
    """
    rfm = aggregate_rfm(logs, base_date)
    for log in rfm:
        r = log["recency"]
        if r <= 30:
            log["r_score"] = 3
        elif r <= 90:
            log["r_score"] = 2
        else:
            log["r_score"] = 1

        f = log["frequency"]
        if f >= 3:
            log["f_score"] = 3
        elif f == 2:
            log["f_score"] = 2
        else:
            log["f_score"] = 1

        m = log["monetary"]
        if m >= 50000:
            log["m_score"] = 3
        elif m >= 10000:
            log["m_score"] = 2
        else:
            log["m_score"] = 1

    return [
        {
            "customer_id": log["customer_id"],
            "r_score": log["r_score"],
            "f_score": log["f_score"],
            "m_score": log["m_score"]
        }
        for log in rfm
    ]

def rank_rfm(logs, base_date):
    """
    Calculate the total score, sort by total score, and assign a rank.
    
    Args:
        logs (list): Customer purchase logs. Each log is a dictionary containing customer_id, 
        date, and amount.
        base_date (datetime): Reference date for calculating recency.
        
    Returns:
        list: A list of dictionaries containing customer_id, total_score, and rank.
    """
    users = score_rfm(logs, base_date)
    for user in users:
        r = user["r_score"]
        f = user["f_score"]
        m = user["m_score"]
        t = r + f + m
        user["total_score"] = t

        if t >= 8:
            user["rank"] = "Champion"
        elif t >= 6:
            user["rank"] = "Loyal"
        elif t >= 4:
            user["rank"] = "At Risk"
        else:
            user["rank"] = "Lost"

    sorted_data = sorted(users, key=lambda x:x["total_score"], reverse=True)

    return [
        {
            "customer_id": user["customer_id"],
            "r_score": user["r_score"],
            "f_score": user["f_score"],
            "m_score": user["m_score"],
            "total_score": user["total_score"],
            "rank": user["rank"]
        }
        for user in sorted_data
    ]

def print_rfm_summary(rfm_data):
    """
    Print r_score, f_score, m_score, total_score, and rank.
    
    Args:
        rfm_data (list): RFM ranking results containing customer_id, scores, total_score, and rank.
    """
    print("Customer ID | R Score | F Score | M Score | Total Score | Rank")

    for data in rfm_data:
        u = data["customer_id"]
        r = data["r_score"]
        f = data["f_score"]
        m = data["m_score"]
        t = data["total_score"]
        rank = data["rank"]
        print(f"{u:<11} | {r:7} | {f:7} | {m:7} | {t:11} | {rank}")

result = rank_rfm(rfm_logs, BASE_DATE)
print_rfm_summary(result)



Customer ID | R Score | F Score | M Score | Total Score | Rank
C006        |       3 |       1 |       3 |           7 | Loyal
C001        |       3 |       2 |       1 |           6 | Loyal
C004        |       3 |       1 |       2 |           6 | Loyal
C005        |       2 |       2 |       2 |           6 | Loyal
C002        |       2 |       1 |       2 |           5 | At Risk
C003        |       1 |       1 |       2 |           4 | At Risk
